# RNN for basic question answer task

In [ ]:
import os
import re
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import pandas as pd

In [ ]:
current_dir = os.getcwd()
root_dir = os.path.dirname(os.path.dirname(current_dir))
dataset_dir = os.path.join(root_dir, "datasets")

In [ ]:
dataset_path = os.path.join(dataset_dir, "QA", "QA_Dataset.csv")
df = pd.read_csv(dataset_path)
df.head(20)

## Tokenize

In [ ]:
def tokenize(text):
    text = text.lower()
    text = re.sub(r"[^a-z0-9]+", " ", text)
    return text.split()

In [ ]:
tokenize('What is the capital of France?')

## Vocabulary

In [ ]:
vocab = {
    "<UNK>": 0
}

In [ ]:
def build_vocab(row):
    tokenized_question = tokenize(row['question'])
    tokenized_answer = tokenize(row['answer'])
    merged_tokens = tokenized_question + tokenized_answer
    for token in merged_tokens:
        if token not in vocab:
            vocab[token] = len(vocab)
    
df.apply(build_vocab, axis=1)

In [ ]:
len(vocab), vocab

## Neumerical Indices

In [ ]:
def text_to_indices(text, vocab):
    indexed_text = []
    for token in tokenize(text):
        if token in vocab:
            indexed_text.append(vocab[token])
        else:
            indexed_text.append(vocab['<UNK>'])
            
    return indexed_text

text_to_indices('What is france doing?', vocab)

# Dataset and Dataloader Creation

In [ ]:
class QADataset(Dataset):
    def __init__(self, df:pd.DataFrame, vocab:dict):
        self.df = df
        self.vocab = vocab
        
    def __len__(self):
        return self.df.shape[0]
    
    def __getitem__(self, index):
        embedded_question = text_to_indices(self.df.iloc[index]['question'], self.vocab)
        embedded_answer = text_to_indices(self.df.iloc[index]['answer'], self.vocab)
        
        return torch.tensor(embedded_question), torch.tensor(embedded_answer)

In [ ]:
dataset = QADataset(df, vocab)

dataloader = DataLoader(dataset, batch_size=1, shuffle=True) 
    # Avoid padding. If batch size was more than 1, padding was needed

In [ ]:
for question, answer in dataloader:
    print(question, answer)

## Model Architecture Development

In [ ]:
class BasicRNN(nn.Module):
    def __init__(self, vocab_size):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embedding_dim=50)
        self.rnn = nn.RNN(50, 64)
        self.fc = nn.Linear(64, vocab_size)
        
    def forward(self):
        pass